# El loop del agente, control de herramientas y auditoría
Ahora vamos a convertir aquel programa en un proyecto más parecido a una aplicación real, que podamos ampliar posteriormente para integrarlo con Zabbix, SSH, Grafana y MCP.

Esta clase tiene cuatro objetivos:

1. Comprender exactamente cómo funciona el loop del agente.
2. Separar el código en módulos Python.
3. Controlar las herramientas que puede ejecutar y limitar sus llamadas.
4. Registrar cada paso del diagnóstico en un fichero de auditoría JSONL.

Al finalizar tendrás un proyecto ejecutable con uv, métricas simuladas y un registro de las decisiones tomadas.

## 1. ¿Qué problema queremos resolver?
Recordemos el bucle de la clase 4:
![Diagrama](img/05_graph.png)

Funciona, pero presenta algunos riesgos:
- ¿Qué ocurre si el LLM solicita 100 herramientas?
- ¿Qué ocurre si intenta consultar un servidor no autorizado?
- ¿Cómo averiguamos qué herramientas se ejecutaron realmente?
- ¿Cómo podemos reconstruir el orden de la investigación después de una incidencia?
Para responder a estas preguntas añadiremos controles deterministas. No dependeremos únicamente de que el LLM obedezca las instrucciones de su prompt.

## 2. Nuestra nueva arquitectura
Separaremos el proyecto en módulos con responsabilidades distintas.

````
ai-ops-agent/
│
├── pyproject.toml
├── main.py
│
├── agent/
│   ├── __init__.py
│   ├── config.py
│   ├── state.py
│   ├── tools.py
│   ├── nodes.py
│   ├── graph.py
│   └── audit.py
│
└── logs/
    └── audit.jsonl       # Se crea automáticamente
````

| Archivo | Responsabilidad |
|---|---|
| `config.py` | Configuración, límites y servidores permitidos. |
| `state.py` | Estado compartido del grafo. |
| `tools.py` | Herramientas para diagnosticar Linux. |
| `nodes.py` | Nodos que invocan el LLM y ejecutan herramientas. |
| `graph.py` | Construcción y conexiones del grafo. |
| `audit.py` | Registro de eventos en JSONL. |
| `main.py` | Punto de entrada del programa. |

La ventaja es que podremos cambiar una implementación sin tener que reescribir todo el agente.


## 3. Primer concepto nuevo: controlar el estado

En la clase 4 utilizamos MessagesState, que almacena el historial de mensajes.
Ahora necesitamos guardar también el número de herramientas solicitadas y el servidor autorizado para esta ejecución.
Lo haremos extendiendo MessagesState:

````python
class AgentState(MessagesState):
    servidor: str
    tool_calls_used: int
````

En términos de Python, MessagesState es un TypedDict. Estamos definiendo un esquema con campos adicionales, no creando un objeto con lógica propia.
Por ejemplo:

````python
{
    "servidor": "srv-oracle-01",
    "tool_calls_used": 2,
    "messages": [...]
}
````

Este estado permite que los nodos compartan información.
En esta clase emplearemos las siguientes reglas:

### Políticas de nuestro laboratorio

| Política | Valor |
|---|---|
| Máximo de llamadas a herramientas | 4 |
| Servidores autorizados | 2 |
| Acceso SSH real | No |
| Operaciones de escritura | No |

Contaremos cada solicitud de ejecución, aunque sea rechazada, para evitar que un modelo agote el bucle intentando repetidamente operaciones inválidas.

## 4. Crear el proyecto

````
uv init ai-ops-agent
cd ai-ops-agent

uv add langgraph langchain-core langchain-openai

mkdir -p agent logs
touch agent/__init__.py
````
Necesitarás Python 3.11 o superior.
Para utilizar un LLM de OpenAI:
````bash
export OPENAI_API_KEY="tu-api-key"
````
En este ejercicio seguimos usando únicamente servidores y métricas ficticios. Si vas a utilizarlo en la empresa, cambiaremos después la integración por el proveedor aprobado, como Azure OpenAI.

## 5. Archivo agent/config.py

Empezaremos por centralizar la configuración.

````python

import os

# Límite total de solicitudes de Tools
MAX_TOOL_CALLS = 4

# Modelo utilizado
MODEL_NAME = os.getenv(
    "OPENAI_MODEL",
    "gpt-4.1-mini"
)

# Laboratorio: servidores autorizados y sus métricas
SERVIDORES = {
    "srv-oracle-01": {
        "cpu": 96,
        "memoria": 72,
        "procesos": [
            {"nombre": "oracle", "cpu": 89},
            {"nombre": "java", "cpu": 4},
            {"nombre": "systemd", "cpu": 1},
        ],
    },
    "srv-linux-ok": {
        "cpu": 15,
        "memoria": 35,
        "procesos": [
            {"nombre": "python", "cpu": 4},
            {"nombre": "systemd", "cpu": 1},
        ],
    },
}

````

En una aplicación real, no guardaríamos métricas de servidores aquí.

Este archivo contendría parámetros como límites, proveedores y configuración; las métricas llegarían desde herramientas SSH, Zabbix o Grafana.

## 6. Archivo agent/state.py

Definiremos nuestro estado:

````python
from langgraph.graph import MessagesState

class AgentState(MessagesState):
    servidor: str
    tool_calls_used: int
````
Aquí aparece una cuestión importante.
Cuando un nodo devuelve:

````python
return {"tool_calls_used": 3}
````

LangGraph sustituye el valor anterior de ese campo.
En cambio, cuando un nodo devuelve:

````python
return {"messages": [respuesta]}
````

Los mensajes se incorporan al historial utilizando el mecanismo de actualización de MessagesState.

Es decir, los distintos campos del estado pueden tener comportamientos de actualización diferentes, conocidos como reducers.

## 7. Archivo agent/tools.py

Vamos a implementar tres herramientas. Son las mismas de la clase anterior, pero separadas en su propio módulo.

````python

from langchain_core.tools import tool

from agent.config import SERVIDORES


def obtener_servidor(servidor: str) -> dict:
    """Obtiene datos simulados de un servidor autorizado."""

    if servidor not in SERVIDORES:
        raise ValueError(
            f"Servidor no autorizado: {servidor}"
        )

    return SERVIDORES[servidor]


@tool
def linux_cpu_usage(servidor: str) -> dict:
    """Consulta el porcentaje de CPU de un servidor
    Linux autorizado del laboratorio."""

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "cpu_percent": datos["cpu"],
    }


@tool
def linux_memory_usage(servidor: str) -> dict:
    """Consulta el porcentaje de memoria utilizada
    por un servidor Linux autorizado del laboratorio."""

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "memory_percent": datos["memoria"],
    }


@tool
def linux_processes(servidor: str) -> dict:
    """Consulta los procesos y su consumo de CPU
    de un servidor Linux autorizado del laboratorio."""

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "procesos": datos["procesos"],
    }


# Catálogo de herramientas autorizadas
TOOLS = [
    linux_cpu_usage,
    linux_memory_usage,
    linux_processes,
]

# Facilita buscar una Tool por su nombre
TOOLS_BY_NAME = {
    herramienta.name: herramienta
    for herramienta in TOOLS
}
````

Observa esta última parte:

````python
TOOLS_BY_NAME = {
    herramienta.name: herramienta
    for herramienta in TOOLS
}
````
Construye un diccionario parecido a:

````python
{
    "linux_cpu_usage": <Tool>,
    "linux_memory_usage": <Tool>,
    "linux_processes": <Tool>
}
````

Nos permitirá buscar una herramienta por el nombre que indique el LLM.
No estamos exponiendo una función que ejecute cualquier comando recibido como texto.
**Solo podrá ejecutar herramientas de este catálogo.**

## 8. Archivo agent/nodes.py

Este es el archivo más importante de la clase.

Contendrá dos nodos:
- agente: llama al LLM.
- ejecutar_tools: valida y ejecuta las herramientas solicitadas.

En la clase 4 utilizábamos directamente ToolNode. Ahora crearemos un ejecutor personalizado para comprender cómo funciona internamente y añadir políticas propias.

````python

import json

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    SystemMessage,
    ToolMessage,
)

from agent.config import (
    MAX_TOOL_CALLS,
    MODEL_NAME,
)
from agent.state import AgentState
from agent.tools import TOOLS, TOOLS_BY_NAME


# ==========================================
# 1. MODELO
# ==========================================

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    timeout=30,
    max_retries=2,
)

llm_with_tools = llm.bind_tools(TOOLS)


# ==========================================
# 2. PROMPT
# ==========================================

SYSTEM_PROMPT = """
Eres un agente de diagnóstico Linux.

Objetivo:
Investigar alertas de CPU de servidores.

Reglas:
- Consulta la CPU mediante herramientas.
- Si está elevada, investiga los procesos.
- Consulta memoria si es relevante.
- No repitas herramientas innecesariamente.
- No inventes métricas ni ejecuciones.
- No ejecutes comandos destructivos.
- Distingue hechos de hipótesis.
- Solo tienes permiso de lectura.
- Responde en español.

Cuando hayas obtenido suficiente información
o se agote el presupuesto de herramientas,
genera el diagnóstico con la información disponible.
Indica qué no has podido comprobar.
"""


# ==========================================
# 3. NODO AGENTE
# ==========================================

def agente(state: AgentState):

    usadas = state["tool_calls_used"]

    mensajes = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ]

    if usadas >= MAX_TOOL_CALLS:
        # El modelo deja de tener Tools disponibles
        respuesta = llm.invoke(
            mensajes + [
                SystemMessage(
                    content=(
                        "Se ha agotado el presupuesto "
                        "de herramientas. Genera ahora "
                        "el diagnóstico final sin "
                        "solicitar otras comprobaciones."
                    )
                )
            ]
        )
    else:
        respuesta = llm_with_tools.invoke(mensajes)

    return {
        "messages": [respuesta]
    }


# ==========================================
# 4. NODO EJECUTOR DE TOOLS
# ==========================================

def ejecutar_tools(state: AgentState):

    ultimo_mensaje = state["messages"][-1]

    usadas = state["tool_calls_used"]
    resultados = []

    for llamada in ultimo_mensaje.tool_calls:

        nombre = llamada["name"]
        argumentos = llamada["args"]
        call_id = llamada["id"]

        if usadas >= MAX_TOOL_CALLS:
            contenido = {
                "error": "Presupuesto de Tools agotado"
            }

        else:
            # Contamos también los intentos rechazados
            usadas += 1

            herramienta = TOOLS_BY_NAME.get(nombre)

            if herramienta is None:
                contenido = {
                    "error": "Tool no autorizada"
                }

            elif not isinstance(argumentos, dict):
                contenido = {
                    "error": "Argumentos inválidos"
                }

            elif argumentos.get("servidor") != state["servidor"]:
                contenido = {
                    "error": "Servidor no autorizado"
                }

            else:
                try:
                    resultado = herramienta.invoke(argumentos)

                    contenido = resultado

                except Exception:
                    # No revelar detalles internos al LLM
                    contenido = {
                        "error": "Error ejecutando Tool"
                    }

        resultados.append(
            ToolMessage(
                content=json.dumps(
                    contenido,
                    ensure_ascii=False
                ),
                name=nombre,
                tool_call_id=call_id,
            )
        )

    return {
        "messages": resultados,
        "tool_calls_used": usadas,
    }

````

### ¿Qué controles hemos implementado?

#### Control 1. Catálogo de herramientas
El nombre solicitado debe existir en TOOLS_BY_NAME. No ejecutamos funciones desconocidas.

#### Control 2. Servidor autorizado
La herramienta solo recibe el servidor que se autorizó al empezar la ejecución.

#### Control 3. Presupuesto de herramientas
Cada solicitud consume presupuesto. Una llamada rechazada también se cuenta.

#### Control 4. Gestión de errores
Una herramienta que falla devuelve un ToolMessage con un error genérico, permitiendo al agente continuar.


Este ejecutor es intencionadamente sencillo. En producción necesitaríamos además validación estricta de argumentos, timeouts de herramientas, autorización por identidad, límites de concurrencia y errores auditados de forma segura.

### ¿Por qué devolvemos ToolMessage?
Porque el LLM necesita recibir el resultado asociado a cada llamada que realizó.
Una solicitud tiene un identificador:

````pyhon
{
    "name": "linux_cpu_usage",
    "args": {"servidor": "srv-oracle-01"},
    "id": "call_123"
}
````
Y nuestro ejecutor responde:

````pyhon
ToolMessage(
    content='{"cpu_percent": 96}',
    tool_call_id="call_123",
    name="linux_cpu_usage"
)
````

El ``tool_call_id`` permite asociar el resultado a la solicitud original.
Esto es especialmente importante cuando el LLM solicita varias herramientas en una misma respuesta.

## 9. Archivo agent/graph.py
Aquí definiremos la arquitectura del grafo.

````python

from typing import Literal

from langgraph.graph import (
    StateGraph,
    START,
    END,
)

from agent.state import AgentState
from agent.nodes import (
    agente,
    ejecutar_tools,
)


# ==========================================
# ROUTER
# ==========================================

def decidir_siguiente_paso(
    state: AgentState,
) -> Literal["ejecutar_tools", "__end__"]:

    ultimo_mensaje = state["messages"][-1]

    if getattr(ultimo_mensaje, "tool_calls", []):
        return "ejecutar_tools"

    return "__end__"


# ==========================================
# CONSTRUIR GRAFO
# ==========================================

builder = StateGraph(AgentState)

builder.add_node("agente", agente)
builder.add_node("ejecutar_tools", ejecutar_tools)

builder.add_edge(START, "agente")

builder.add_conditional_edges(
    "agente",
    decidir_siguiente_paso,
    {
        "ejecutar_tools": "ejecutar_tools",
        "__end__": END,
    },
)

# LOOP
builder.add_edge("ejecutar_tools", "agente")

graph = builder.compile()

````

Fíjate en lo poco que necesita saber el archivo graph.py sobre las herramientas.
No conoce cómo se consulta la CPU ni qué modelo de LLM utilizamos. Solo conoce los nodos, su estado y las conexiones.
Eso es precisamente lo que queremos: **separar orquestación de implementación.**

## 10. Registrar los pasos del diagnóstico
Esta parte es especialmente interesante para un agente de operaciones.
Imagina que un usuario informa:
El agente ha diagnosticado mal una alerta de CPU.

Necesitaremos reconstruir qué ocurrió.
Por ejemplo:
`````
20:40:01  START          srv-oracle-01
20:40:02  LLM            solicita linux_cpu_usage
20:40:02  TOOL           linux_cpu_usage OK
20:40:04  LLM            solicita linux_processes
20:40:04  TOOL           linux_processes OK
20:40:06  LLM            genera diagnóstico
20:40:06  END            2 solicitudes
````
Para hacerlo utilizaremos dos elementos:
- graph.stream(): permite observar la ejecución paso a paso.
- Un archivo JSONL: guarda un objeto JSON por línea.
LangGraph permite obtener actualizaciones por nodo con stream_mode="updates", en lugar de esperar al resultado final. 

Docs by LangChain



Archivo agent/audit.py

````python

import json
from datetime import datetime, timezone
from pathlib import Path


LOG_FILE = Path("logs/audit.jsonl")


def registrar_evento(
    run_id: str,
    servidor: str,
    evento: str,
    detalle: dict | None = None,
):
    """Escribe un evento estructurado en JSONL."""

    LOG_FILE.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    registro = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "run_id": run_id,
        "servidor": servidor,
        "evento": evento,
        "detalle": detalle or {},
    }

    with LOG_FILE.open(
        "a",
        encoding="utf-8"
    ) as fichero:

        fichero.write(
            json.dumps(
                registro,
                ensure_ascii=False
            ) + "\n"
        )

````
